# Capstone: the whole desk

This notebook combines the course into one held-out run.

The desk will:

- read shop data through MCP;
- consult a supplier peer through A2A;
- use a tool agent;
- trace model and tool work;
- limit each ticket with a budget;
- gate risky actions;
- verify the final decision; and
- score all eight test tickets.

Before running this notebook, install both protocol extras once with: pip install -e ".[mcp,a2a]".

In [ ]:
import asyncio
import json
import os
import sys
import threading
from collections import Counter
from getpass import getpass
from pathlib import Path

import litellm
from dotenv import load_dotenv

course_root = Path.cwd()
if not (course_root / "pyproject.toml").exists():
    course_root = course_root.parent
load_dotenv(course_root / ".env")
if not os.getenv("OPENROUTER_API_KEY"):
    os.environ["OPENROUTER_API_KEY"] = getpass("OpenRouter API key: ")

MODEL = os.getenv("MODEL", "openrouter/deepseek/deepseek-v3.2")
litellm.drop_params = True
litellm.cache = litellm.Cache(type="disk", disk_cache_dir=str(course_root / ".litellm_cache"))

## 1. Load the held-out queue

The test split was not used to build or tune the earlier examples. We keep its gold answers only for scoring after each run.

In [ ]:
from shoplab import world
from shoplab.trace import Tracer

TEST = world.load_tickets()["test"]
orders = {item["order_id"]: item for item in world.load_orders()}
customers = {item["customer_id"]: item for item in world.load_customers()}
tracer = Tracer()

for ticket in TEST:
    print(ticket["ticket_id"], "->", ticket["gold"])

## 2. Read shop data through MCP

This clear learning adapter opens a short MCP session for each tool call. A production client would normally keep one connection open.

In [ ]:
def run_async_in_thread(async_function):
    box = {}

    def worker():
        try:
            box["value"] = asyncio.run(async_function())
        except Exception as error:
            box["error"] = error

    thread = threading.Thread(target=worker)
    thread.start()
    thread.join()

    if "error" in box:
        raise box["error"]
    return box["value"]

In [ ]:
from mcp import ClientSession, StdioServerParameters
from mcp.client.stdio import stdio_client
from shoplab.tools import Tool

MCP_SERVER = course_root / "mcp_servers/shopdesk_server.py"

def mcp_params():
    return StdioServerParameters(
        command=sys.executable,
        args=[str(MCP_SERVER)],
        env=os.environ.copy(),
    )

async def mcp_list_tools():
    async with stdio_client(mcp_params()) as (reader, writer):
        async with ClientSession(reader, writer) as session:
            await session.initialize()
            return (await session.list_tools()).tools

async def mcp_call(name, arguments):
    async with stdio_client(mcp_params()) as (reader, writer):
        async with ClientSession(reader, writer) as session:
            await session.initialize()
            result = await session.call_tool(name, arguments)
            if result.isError:
                return {"error": result.content[0].text}
            if result.structuredContent is not None:
                value = result.structuredContent
                return value.get("result", value) if isinstance(value, dict) else value
            return json.loads(result.content[0].text)

remote_specs = run_async_in_thread(mcp_list_tools)
mcp_tools = {}

for spec in remote_specs:
    def call_remote(_name=spec.name, **arguments):
        return run_async_in_thread(lambda: mcp_call(_name, arguments))

    mcp_tools[spec.name] = Tool(
        spec.name,
        spec.description or "MCP tool",
        spec.inputSchema,
        call_remote,
    )

print("MCP tools:", sorted(mcp_tools))
print("sample order status:", mcp_tools["get_order"].fn(order_id="ORD-7301")["status"])

## 3. Consult the supplier through A2A

We start the supplier peer, discover its card, and request one restock quote. The quote becomes available to the desk as a small tool result.

In [ ]:
import httpx
from a2a.client import A2ACardResolver, ClientConfig, ClientFactory
from a2a.helpers import get_stream_response_text, new_text_message
from a2a.types import Role, SendMessageRequest
from a2a_servers.supplier_agent import serve_in_background

supplier_server = serve_in_background()
supplier_url = supplier_server.__enter__()
supplier_http = httpx.AsyncClient(timeout=10)
supplier_card = await A2ACardResolver(
    httpx_client=supplier_http,
    base_url=supplier_url,
).get_agent_card()
supplier_client = ClientFactory(
    ClientConfig(httpx_client=supplier_http, streaming=True)
).create(supplier_card)

async def send_supplier(text, task_id=None, context_id=None):
    message = new_text_message(
        text,
        role=Role.ROLE_USER,
        task_id=task_id,
        context_id=context_id,
    )
    answer = ""
    state = None
    async for event in supplier_client.send_message(SendMessageRequest(message=message)):
        kind = event.WhichOneof("payload")
        if kind == "task":
            task_id = event.task.id
            context_id = event.task.context_id
            state = event.task.status.state
        elif kind == "status_update":
            task_id = event.status_update.task_id
            context_id = event.status_update.context_id
            state = event.status_update.status.state
        answer = get_stream_response_text(event) or answer
    return task_id, context_id, state, answer

low_stock = next(
    item
    for item in world.load_products()
    if item["stock"] <= item["reorder_threshold"]
)
supplier_task, supplier_context, _, question = await send_supplier(
    f"Restock lead time for {low_stock['sku']}?"
)
_, _, _, supplier_quote = await send_supplier(
    "west",
    task_id=supplier_task,
    context_id=supplier_context,
)

print("peer:", supplier_card.name)
print("quote:", supplier_quote)

In [ ]:
def ask_supplier(sku):
    return {
        "sku": sku,
        "region": "west",
        "quote": supplier_quote,
    }

supplier_tool = Tool(
    "ask_supplier",
    "Return the supplier quote fetched over A2A.",
    {
        "type": "object",
        "properties": {"sku": {"type": "string"}},
        "required": ["sku"],
    },
    ask_supplier,
)

print(supplier_tool.fn(low_stock["sku"]))

## 4. Assemble the controlled desk

Each ticket gets fresh budget and side-effect state. Read tools come from MCP. Risky local tools pass through approval. A free rule check runs when the model tries to finish.

In [ ]:
import shoplab.llm
from shoplab.controls import Budget, BudgetExceeded, require_approval
from shoplab.evals import score_ticket
from shoplab.loop import run_agent
from shoplab.tools import Ledger, standard_tools
from shoplab.verify import check_decision

def traced(tool):
    def call(**arguments):
        with tracer.span(tool.name, kind="tool"):
            return tool.fn(**arguments)
    return Tool(tool.name, tool.description, tool.params, call, risky=tool.risky)

def approval_for(ledger):
    def approve(name, arguments):
        if name != "issue_refund":
            return True
        amount = arguments.get("amount_usd", 0) or 0
        if amount < 200:
            return True
        ledger.record("approval_escalation", amount_usd=amount)
        return False
    return approve

SYSTEM = (
    "Clear this Larkspur return. Get order, customer, and policy through the read tools. "
    "Use calc for money. Carry out refund, replacement, or escalation, then call finish "
    "with decision, policy_id, and refund_usd. Do not repeat lookups."
)

In [ ]:
def clear_ticket(ticket):
    ledger = Ledger()
    budget = Budget(max_calls=16, max_cost_usd=0.05)
    local = standard_tools(ledger)

    tools = {
        name: mcp_tools[name]
        for name in ("get_order", "get_customer", "search_policy", "check_inventory")
    }
    tools["ask_supplier"] = supplier_tool
    for name in ("calc", "escalate", "finish"):
        tools[name] = local[name]

    approve = approval_for(ledger)
    for name in ("issue_refund", "create_replacement"):
        tools[name] = require_approval(local[name], approve)

    tools = {name: traced(tool) for name, tool in tools.items()}
    verification = {"result": None}

    def before_tool(name, arguments):
        if name == "finish":
            verification["result"] = check_decision(
                arguments,
                ticket,
                orders[ticket["order_id"]],
                customers[ticket["customer_id"]],
            )
        return True

    def charge(step, message):
        budget.charge(shoplab.llm.LEDGER[-1]["cost_usd"])

    task = json.dumps({key: value for key, value in ticket.items() if key != "gold"})

    try:
        with tracer.span("triage", kind="agent", ticket=ticket["ticket_id"]):
            result = run_agent(
                task,
                tools,
                model=MODEL,
                system=SYSTEM,
                max_steps=12,
                before_tool=before_tool,
                on_step=charge,
            )
        answer = result.answer if isinstance(result.answer, dict) else {}
        stop = result.stop_reason
    except BudgetExceeded as error:
        answer = {}
        stop = "budget: " + str(error)

    return {
        "ticket": ticket,
        "prediction": answer,
        "score": score_ticket(answer, ticket["gold"]),
        "verification": verification["result"],
        "actions": ledger.entries,
        "budget": budget.snapshot(),
        "stop": stop,
    }

## 5. Clear and score the test queue

This cell makes the full held-out run. It may take several minutes because each read tool uses a clear, short-lived MCP session.

In [ ]:
runs = [clear_ticket(ticket) for ticket in TEST]

for run in runs:
    check = run["verification"]
    print(
        run["ticket"]["ticket_id"],
        "pred:", run["prediction"].get("decision"),
        "gold:", run["ticket"]["gold"]["decision"],
        "exact:", run["score"]["exact"],
        "verified:", check["agrees"] if check else None,
        "stop:", run["stop"],
    )

def accuracy(field):
    return sum(run["score"][field] for run in runs) / len(runs)

print("\ndecision accuracy:", round(accuracy("decision_correct"), 3))
print("policy accuracy:", round(accuracy("policy_correct"), 3))
print("amount accuracy:", round(accuracy("amount_correct"), 3))
print("exact accuracy:", round(accuracy("exact"), 3))
print("usage:", shoplab.llm.usage_summary())

## 6. Read the trace

The trace shows one agent span for each ticket and child spans for its tools.

In [ ]:
tracer.print_tree()

print("\nspan counts:")
for (kind, name), count in sorted(Counter((span.kind, span.name) for span in tracer.spans).items()):
    print(kind, name, count)

In [ ]:
try:
    import obs
    from shoplab.trace import export_phoenix

    obs.enable_phoenix()
    print("Phoenix export:", export_phoenix(tracer))
except Exception as error:
    print("Phoenix skipped:", error)

## 7. Tear down external resources

The MCP sessions already closed after each call. We still need to close the A2A HTTP client and supplier server.

In [ ]:
supplier_server.__exit__(None, None, None)
await supplier_http.aclose()

print("supplier stopped; MCP sessions closed")

## What you built

1. MCP provided remote read tools.
2. A2A provided a supplier peer answer.
3. The agent loop chose and used tools.
4. A budget limited each ticket.
5. Approval guarded risky side effects.
6. Verification checked each final decision.
7. Evaluation scored the held-out test split.
8. Tracing showed the run as a tree.

The course ends where production work begins: improve the failed rows, rerun the same held-out evaluation, and keep every control in place.

## Small practice steps

1. Run only TEST[:2] while changing the system prompt.
2. Lower the automatic refund threshold.
3. Pick one failed ticket and inspect its tool spans.